# 5/7 - G/H. Summary + event registry
Team pressing profile, diagnostics, and every metric's events mapped to one standard event table (the input to the clip planner).

*Original sections: sections 17, 19, 19b.*  Barca (team 0) vs Atletico (team 1), first half.

**Run the notebooks in order** (each one reads what the earlier ones saved in `new_cache/pressure_analysis/state/`):

| # | notebook | what it covers |
|---|---|---|
| 1 | 01_pressure_signal.ipynb | B. The pressure signal |
| 2 | 02_where_zone_channel_line.ipynb | C. WHERE - zone, channel, defensive line, shape |
| 3 | 03_timing_and_possession.ipynb | D. WHEN - reaction speed and possession-level pressure |
| 4 | 04_passes_outcomes_triggers.ipynb | E/F. EFFECT and TRIGGERS |
| 5 | **05_summary_and_registry.ipynb** | G/H. Summary + event registry |
| 6 | 06_clips.ipynb | H. Clips |
| 7 | 07_exports.ipynb | I. Exports |

Shared setup (paths, tables, data contract, team-label orientation, helpers) lives in `pressure_common.py` - keep it in the same folder as the notebooks.

In [3]:
from pressure_common import *

In [4]:
# handed over by the earlier notebooks:
#   03_timing_and_possession: COUNTER_PRESS_WINDOW_FRAMES, MAX_REGAIN_SEC, final_pressure_by_team, possession_events, pressing_intensity_timeline, regain_summary, summary, team_pressure_events, turnovers_annotated
#   01_pressure_signal: FRAC_TIGHT_BLOCKED_THRESH, PRESSURE_DIST_THRESH_M, STOPPAGE_GAP_SEC, off_ball_pressure, stoppages, tpf
#   04_passes_outcomes_triggers: PRESS_ONSET_WINDOW_FRAMES, carrier_runs_full, defensive_actions, long_gap_events, pass_summary, passes_from_scratch, ppda_by_team, ppda_detail, reception_triggers, zone_summary
#   02_where_zone_channel_line: defensive_line_events, line_state, team_shape
load_state(globals(), [
    'COUNTER_PRESS_WINDOW_FRAMES',
    'FRAC_TIGHT_BLOCKED_THRESH',
    'MAX_REGAIN_SEC',
    'PRESSURE_DIST_THRESH_M',
    'PRESS_ONSET_WINDOW_FRAMES',
    'STOPPAGE_GAP_SEC',
    'carrier_runs_full',
    'defensive_actions',
    'defensive_line_events',
    'final_pressure_by_team',
    'line_state',
    'long_gap_events',
    'off_ball_pressure',
    'pass_summary',
    'passes_from_scratch',
    'possession_events',
    'ppda_by_team',
    'ppda_detail',
    'pressing_intensity_timeline',
    'reception_triggers',
    'regain_summary',
    'stoppages',
    'summary',
    'team_pressure_events',
    'team_shape',
    'tpf',
    'turnovers_annotated',
    'zone_summary',
])

loaded 28 variables from C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache\pressure_analysis\state


## 17. Team pressing profile -- summary
One table per team pulling together volume (section 4/12), effectiveness by zone (15b), counter-press response speed (8b), and reception triggers (16) -- everything an LLM narrative layer needs about a team's pressing identity in one place, instead of six separate printouts.

In [5]:
# TEAM_NAMES comes from Setup-2 (derived from the table's team_name column)

profile_rows = []
for team_id, opponent_id in [(0, 1), (1, 0)]:
    row = {"team": TEAM_NAMES[team_id]}

    # volume + regain (section 12 / section 14) -- both indexed by the PRESSING team
    row["pct_opp_possession_pressured"] = (
        final_pressure_by_team.loc[team_id, "duration_weighted_pct_under_pressure"]
        if team_id in final_pressure_by_team.index else np.nan
    )
    row["ppda"] = ppda_by_team.get(team_id, np.nan)
    row["press_event_regain_rate_pct"] = (
        regain_summary.loc[team_id, "regain_rate"] if team_id in regain_summary.index else np.nan
    )

    # outcome/turnover effectiveness by zone (section 15b) -- indexed by the team BEING pressured,
    # so this team's press effectiveness reads from the opponent's row. zone_summary's "zone" is
    # relative to the team whose action it is (the opponent here) -- e.g. the opponent's own_third
    # is physically THIS (pressing) team's attacking_third, not its own_third. Bug found via the
    # LLM report cross-check: the old version reused the same zone label on both sides, so
    # "Barca turnover_rate_own_third" actually held Atletico-in-Atletico's-own-third (the correct
    # NUMBER, attached to a backwards column NAME). ZONE_FLIP maps a zone label from this team's
    # own perspective to the physically-equivalent label from the opponent's perspective.
    ZONE_FLIP = {"own_third": "attacking_third", "middle_third": "middle_third", "attacking_third": "own_third"}
    for zone in ["own_third", "middle_third", "attacking_third"]:
        key = (opponent_id, ZONE_FLIP[zone])
        row[f"turnover_rate_{zone}"] = zone_summary.loc[key, "turnover_rate"] if key in zone_summary.index else np.nan

    # counter-press: this team's own regain rate when THEY pressed fast after losing the ball (section 8b)
    fast_row = summary[(summary["losing_team"] == team_id) & (summary["press_response"] == "fast")]
    row["counter_press_fast_regain_rate_pct"] = (
        fast_row["regain_rate_pct"].values[0] if len(fast_row) and fast_row["turnovers"].values[0] > 0 else np.nan
    )

    # baseline: how often does this team's press actually follow an opponent's reception at all (section 16)
    opp_receptions = reception_triggers[reception_triggers["team"] == opponent_id]
    row["press_follow_rate_on_opp_receptions_pct"] = (
        opp_receptions["press_followed"].mean() * 100 if len(opp_receptions) else np.nan
    )

    profile_rows.append(row)

team_pressing_profile = pd.DataFrame(profile_rows).set_index("team").round(1)
print(team_pressing_profile.T)

team                                     Barca  Atletico
pct_opp_possession_pressured              19.2      16.0
ppda                                       2.2       5.3
press_event_regain_rate_pct               47.0      16.0
turnover_rate_own_third                   57.1      42.9
turnover_rate_middle_third                31.4      23.5
turnover_rate_attacking_third             44.4      50.0
counter_press_fast_regain_rate_pct        33.3      14.3
press_follow_rate_on_opp_receptions_pct   54.5      42.4


In [6]:
# --- one-paragraph auto-summary per team, the shape an LLM prompt actually wants ---
for team_id in [0, 1]:
    name = TEAM_NAMES[team_id]
    r = team_pressing_profile.loc[name]
    print(f"{name}:")
    print(
        f"  Pressed the opponent for {r['pct_opp_possession_pressured']:.1f}% of their out-of-possession time "
        f"(PPDA {r['ppda']:.2f}), regaining the ball within 5s of a press event {r['press_event_regain_rate_pct']:.1f}% of the time."
    )
    print(
        f"  Turnover rate when pressing: {r['turnover_rate_own_third']:.1f}% (own third) / "
        f"{r['turnover_rate_middle_third']:.1f}% (middle) / {r['turnover_rate_attacking_third']:.1f}% (attacking third)."
    )
    print(
        f"  When they pressed within 5s of losing the ball themselves, regained it {r['counter_press_fast_regain_rate_pct']:.1f}% of the time."
    )
    print()

Barca:
  Pressed the opponent for 19.2% of their out-of-possession time (PPDA 2.20), regaining the ball within 5s of a press event 47.0% of the time.
  Turnover rate when pressing: 57.1% (own third) / 31.4% (middle) / 44.4% (attacking third).
  When they pressed within 5s of losing the ball themselves, regained it 33.3% of the time.

Atletico:
  Pressed the opponent for 16.0% of their out-of-possession time (PPDA 5.30), regaining the ball within 5s of a press event 16.0% of the time.
  Turnover rate when pressing: 42.9% (own third) / 23.5% (middle) / 50.0% (attacking third).
  When they pressed within 5s of losing the ball themselves, regained it 14.3% of the time.



## 19. Diagnostic summary
Run after executing the full notebook top to bottom. Prints the key checkpoints from every section so results can be sanity-checked in one place.

In [7]:
print("=" * 60)
print("0. STOPPAGES")
print("=" * 60)
print(f"{len(stoppages)} stoppages >= {STOPPAGE_GAP_SEC}s flagged, "
      f"{stoppages['duration_sec'].sum():.1f}s total dead time excluded from live-play calculations")

print("=" * 60)
print("1. POSSESSION EVENTS")
print("=" * 60)
print(f"{len(possession_events)} possession events (keep-last gap fill)")
print(possession_events["duration_sec"].describe())

print("\n" + "=" * 60)
print("2. PER-EVENT PRESSURE MATCH + FINAL AVERAGE")
print("=" * 60)
matched = possession_events["pct_frames_under_pressure"].notna().sum()
print(f"pressure computed for {matched} / {len(possession_events)} events "
      f"({matched / len(possession_events) * 100:.1f}%)")
print(final_pressure_by_team)

print("\n" + "=" * 60)
print("2b. PRESSING SIGNAL SANITY (gating, section 1)")
print("=" * 60)
print("share of off-ball rows within 5m that got GATED OUT (passive block, not real pressure):")
close_range = off_ball_pressure[off_ball_pressure["dist_to_carrier"] < PRESSURE_DIST_THRESH_M]
print(f"{(~close_range['is_pressuring']).mean() * 100:.1f}% of close-range rows excluded by the gate")
print("\nis_pressuring rate by team (close-range rows only):")
print(close_range.groupby("team")["is_pressuring"].mean())

print("\n" + "=" * 60)
print("3. PASS DETECTION")
print("=" * 60)
print(f"{len(passes_from_scratch)} pass attempts detected (from {len(carrier_runs_full)} carrier runs)")
print(passes_from_scratch["outcome"].value_counts())
print("\npass_distance_m stats:")
print(passes_from_scratch["pass_distance_m"].describe())
print(f"\n{len(defensive_actions)} defensive actions captured (GK saves excluded)")
print(defensive_actions["action_type"].value_counts())

print("\n" + "=" * 60)
print("4. PASSES UNDER PRESSURE")
print("=" * 60)
print("counts by team / pressure state:")
print(passes_from_scratch.groupby(["passer_team", "under_pressure"]).size())
print("\nsuccess rate table:")
print(pass_summary)

print("\n" + "=" * 60)
print("5. PPDA")
print("=" * 60)
print("passes allowed / defensive actions made, by team (excluding each team's own defensive third):")
for team_id in [0, 1]:
    print(f"team {team_id}: {ppda_detail[team_id]}")
print("\nPPDA by team (lower = more aggressive pressing):")
for team_id, val in ppda_by_team.items():
    print(f"team {team_id}: {val:.2f}")

print("\n" + "=" * 60)
print("6. PRESSURE EFFECTIVENESS (co-press, time bins, regain rate)")
print("=" * 60)
print("max simultaneous pressers per event, by team:")
print(team_pressure_events.groupby("team")["max_n_pressing"].describe())
print("\nshare of events that were a coordinated press (max_n_pressing >= 2), by team:")
print(team_pressure_events.groupby("team")["max_n_pressing"].apply(lambda s: (s >= 2).mean() * 100).round(1))
print("\npressing intensity (% of out-of-possession time under pressure) by 5-minute bin:")
print(pressing_intensity_timeline)
print(f"\npressure -> regain conversion rate (within {MAX_REGAIN_SEC}s of event end):")
print(regain_summary)

0. STOPPAGES
25 stoppages >= 10.0s flagged, 436.0s total dead time excluded from live-play calculations
1. POSSESSION EVENTS
218 possession events (keep-last gap fill)
count    218.000000
mean      10.901651
std       14.710322
min        0.360000
25%        2.090000
50%        5.720000
75%       14.230000
max      134.240000
Name: duration_sec, dtype: float64

2. PER-EVENT PRESSURE MATCH + FINAL AVERAGE
pressure computed for 218 / 218 events (100.0%)
               duration_weighted_pct_under_pressure  \
pressing_team                                         
0                                         19.172320   
1                                         15.954116   

               simple_mean_pct_under_pressure  n_events  total_duration_sec  
pressing_team                                                                
0                                   22.842153     111.0             1002.56  
1                                   19.024066     107.0             1374.00  

2b. PRESSI

## 19b. Event registry - every metric in this notebook as one standard event table
Each analysis above produces its own table with its own columns. Here they are all mapped onto **one schema** so they can be listed, filtered, ranked and clipped the same way:

`uid, family, subtype, team, start_frame, end_frame, anchor_frame, zone, channel, target, score, clock, payload`

- `team` is always the **focus team** - the one doing the pressing / defending (so every clip is shown from its point of view).
- `zone` / `channel` are in that focus team's own frame (same convention as sections 5 and 5b); `target` = who the press was applied on (goalkeeper / centre-back / wide defender / midfielder / advanced player).
- `score` ranks events inside a family (the clip planner takes the best ones).

| family | source | subtypes |
|---|---|---|
| `pressure_event` | section 3 + 14 | coordinated_press, lone_chaser, lane_block_press |
| `pressure_regain` | section 14 | press_won_ball |
| `defensive_line` | section 6b | high_line, deep_line, line_step_up |
| `team_shape` | section 18 | compact_block, stretched_block |
| `counter_press` | section 8b | fast_regain, fast_no_regain, slow, never |
| `pass_under_pressure` | section 10 / 15 | forced_backward, forced_sideways, lost_intercepted, progressed_forward (press beaten) |
| `long_ball` | section 15 | lost_after_long_ball |
| `defensive_action` | section 10 / 13 | tackle, interception |
| `press_trigger` | section 16 | touchline, deep_own_third, backward_pass, isolated, static_touch |
| `possession_pressure` | section 12 | heavily_pressed, lightly_pressed |


In [8]:
# ===== 19b. Event registry =====
MIN_SHAPE_EVENT_SEC = 3.0
MIN_POSSESSION_EVENT_SEC = 4.0
_ball_xy = ball_frame_table.set_index("frame_idx")[["ball_pitch_x", "ball_pitch_y"]]
_car_tbl = ball_frame_table.set_index("frame_idx")[["carrier_track_id", "carrier_team"]]
_n_press_lookup = tpf.set_index(["frame_idx", "team"])["n_pressing"]
ZONE_W = {"own_third": 1, "middle_third": 2, "attacking_third": 3}

def _ball_at(f0, f1):
    seg = _ball_xy.loc[f0:f1].dropna()
    return (float(seg["ball_pitch_x"].median()), float(seg["ball_pitch_y"].median())) if len(seg) else (np.nan, np.nan)

def _first_carrier(f0, f1):
    seg = _car_tbl.loc[f0:f1].dropna(subset=["carrier_track_id"])
    return (int(seg["carrier_track_id"].iloc[0]), int(seg["carrier_team"].iloc[0])) if len(seg) else (None, None)

def _press_target(cid, cteam, x, y):
    # who is being pressed: classified in the CARRIER's own frame
    if cid is None or pd.isna(x):
        return "unknown"
    if TRACK_ROLE.get(int(cid)) == "goalkeeper":
        return "goalkeeper"
    z = zone_for_team(x, cteam); ch = channel_for_team(y, cteam)
    if z == "own_third":
        return "centre-back" if ch == "centre" else "wide defender"
    return "midfielder" if z == "middle_third" else "advanced player"

_rows = []
def _add(family, subtype, team, f0, f1, anchor, bx_, by_, score, target=None, **payload):
    team = int(team)
    _rows.append(dict(family=family, subtype=subtype, team=team, start_frame=int(f0), end_frame=int(f1), anchor_frame=int(anchor),
                      zone=zone_for_team(bx_, team) if pd.notna(bx_) else None, channel=channel_for_team(by_, team) if pd.notna(by_) else None,
                      target=target, score=float(score) if pd.notna(score) else 0.0, payload=payload))

# 1) pressure events (+ the ones that won the ball back)
for r in team_pressure_events.itertuples():
    f0, f1 = int(r.start_frame), int(r.end_frame); bx, by = _ball_at(f0, f1)
    cid, cteam = _first_carrier(f0, f1); sx, sy = _ball_at(f0, f0 + 5)
    tgt = _press_target(cid, cteam if cteam is not None else 1 - int(r.team), sx, sy)
    maxn = 0 if pd.isna(r.max_n_pressing) else float(r.max_n_pressing)
    sub = "coordinated_press" if maxn >= 2 else ("lane_block_press" if r.mean_frac_tight_blocked >= FRAC_TIGHT_BLOCKED_THRESH else "lone_chaser")
    common = dict(max_n=maxn, mean_n=float(r.mean_n_pressing), regained=bool(r.regained_ball), duration_sec=float(r.duration_sec))
    _add("pressure_event", sub, r.team, f0, f1, f0, bx, by, r.duration_sec * max(maxn, 1), tgt, **common)
    if bool(r.regained_ball):
        _add("pressure_regain", "press_won_ball", r.team, f0, f1 + COUNTER_PRESS_WINDOW_FRAMES, f1, bx, by, r.duration_sec * max(maxn, 1) + 5, tgt, **common)

# 2) defensive line events (6b)
for r in defensive_line_events.itertuples():
    bx, by = _ball_at(r.start_frame, r.end_frame)
    sc = r.rise_m if r.subtype == "line_step_up" else r.duration_sec * abs(r.mean_line_m - line_state.loc[line_state.team == r.team, "line_last3_m"].median())
    _add("defensive_line", r.subtype, r.team, r.start_frame, r.end_frame, r.start_frame, bx, by, sc,
         mean_line_m=float(r.mean_line_m), peak_line_m=float(r.peak_line_m), rise_m=None if pd.isna(r.rise_m) else float(r.rise_m))

# 3) team shape events (18): sustained compact / stretched blocks
for _t, _g in team_shape.groupby("team"):
    _g = _g.sort_values("frame_idx"); fr_ = _g["frame_idx"].values; a_ = _g["compactness_area_m2"].values
    lo, hi = np.percentile(a_, 10), np.percentile(a_, 90)
    for sub, mask in [("compact_block", a_ <= lo), ("stretched_block", a_ >= hi)]:
        for a, b in mask_runs(fr_, mask, max_gap=5, min_len=int(MIN_SHAPE_EVENT_SEC * FPS)):
            seg = _g[(_g["frame_idx"] >= a) & (_g["frame_idx"] <= b)]; bx, by = _ball_at(a, b)
            mean_area = seg["compactness_area_m2"].mean()
            _add("team_shape", sub, _t, a, b, a, bx, by, (b - a) / FPS * abs(mean_area - np.median(a_)) / 100,
                 depth_m=float(seg["depth_m"].mean()), width_m=float(seg["width_m"].mean()), area_m2=float(mean_area))

# 4) counter-press (8b): the team that LOST the ball is the focus team
for r in turnovers_annotated.itertuples():
    resp, regained = str(r.press_response), bool(r.regained_within_window)
    sub = (f"fast_{'regain' if regained else 'no_regain'}" if resp == "fast" else resp)
    bx, by = _ball_at(int(r.gain_frame), int(r.gain_frame) + 5)
    tt = r.time_to_press_sec
    sc = {"fast": 100 - (0 if pd.isna(tt) else tt * 10) + (20 if regained else 0), "slow": 0 if pd.isna(tt) else tt, "never": 0}.get(resp, 0)
    _add("counter_press", sub, r.losing_team, int(r.loss_frame), int(r.gain_frame) + COUNTER_PRESS_WINDOW_FRAMES, int(r.gain_frame), bx, by, sc,
         loss_frame=int(r.loss_frame), gain_frame=int(r.gain_frame), time_to_press_sec=None if pd.isna(tt) else float(tt), regained=regained, response=resp)

# 5) passes under pressure (10 / 15): focus = the PRESSING team = opponent of the passer
for r in passes_from_scratch[passes_from_scratch["under_pressure"]].itertuples():
    focus = 1 - int(r.passer_team); npr = _n_press_lookup.get((int(r.release_frame), focus), 0)
    _add("pass_under_pressure", str(r.pass_outcome_detail), focus, int(r.release_frame) - 15, int(r.reception_frame) + 10, int(r.release_frame),
         r.release_x, r.release_y if hasattr(r, "release_y") else np.nan, float(npr) + (3 if str(r.pass_outcome_detail) == "lost_intercepted" else 0),
         passer_track_id=int(r.passer_track_id), receiver_track_id=int(r.receiver_track_id), release_frame=int(r.release_frame),
         reception_frame=int(r.reception_frame), outcome=str(r.pass_outcome_detail), pass_distance_m=float(r.pass_distance_m),
         progression_m=None if pd.isna(r.progression_m) else float(r.progression_m), n_pressers=float(npr))

# 6) long balls lost under pressure (15)
if "frame" in long_gap_events.columns:
    for r in long_gap_events[(long_gap_events["under_pressure"] == True) & (long_gap_events["pass_outcome_detail"] == "lost_after_long_ball")].itertuples():
        bx, by = _ball_at(int(r.frame), int(r.frame) + 5)
        _add("long_ball", "lost_after_long_ball", 1 - int(r.team), int(r.frame) - 25, int(r.reception_frame) + 15, int(r.frame), bx, by, 1.0,
             release_frame=int(r.frame), reception_frame=int(r.reception_frame))

# 7) tackles + interceptions (10 / 13): focus = the team that won the ball
for r in defensive_actions.itertuples():
    f = int(r.frame)
    _add("defensive_action", str(r.action_type), r.defending_team, f - 25, f + 25, f, r.x, r.y,
         ZONE_W.get(zone_for_team(r.x, int(r.defending_team)), 1), action_frame=f, action_type=str(r.action_type), x=float(r.x), y=float(r.y))

# 8) press triggers (16)
TRIGGER_RULES = {"trigger_touchline": ("touchline", lambda r: -r.dist_to_touchline_m),
                 "trigger_deep_own_third": ("deep_own_third", lambda r: -(r.receiver_x if r.team == 0 else PITCH_LENGTH - r.receiver_x)),
                 "trigger_backward_pass": ("backward_pass", lambda r: -r.progression_m),
                 "trigger_isolated": ("isolated", lambda r: -(r.n_lanes_open if pd.notna(r.n_lanes_open) else 99)),
                 "trigger_static_touch": ("static_touch", lambda r: -(r.first_touch_speed_mps if pd.notna(r.first_touch_speed_mps) else 99))}
for r in reception_triggers.itertuples():
    focus = 1 - int(r.team)
    for col, (name, scorer) in TRIGGER_RULES.items():
        if bool(getattr(r, col)):
            _add("press_trigger", name, focus, int(r.reception_frame) - 25, int(r.reception_frame) + PRESS_ONSET_WINDOW_FRAMES + 25, int(r.reception_frame),
                 r.receiver_x, r.receiver_y, scorer(r) + (5 if r.press_followed else 0), receiver_track_id=int(r.receiver_track_id),
                 reception_frame=int(r.reception_frame), trigger=name, press_followed=bool(r.press_followed), receiver_team=int(r.team))

# 9) possessions: heavily vs lightly pressed (12)
for r in possession_events.dropna(subset=["pct_frames_under_pressure"]).itertuples():
    if r.duration_sec < MIN_POSSESSION_EVENT_SEC:
        continue
    bx, by = _ball_at(int(r.start_frame), int(r.end_frame))
    if r.pct_frames_under_pressure >= 0.5:
        _add("possession_pressure", "heavily_pressed", r.pressing_team, int(r.start_frame), int(r.end_frame), int(r.start_frame), bx, by,
             r.pct_frames_under_pressure * np.sqrt(r.duration_sec), pct_pressed=float(r.pct_frames_under_pressure), possession_team=int(r.possession_team))
    elif r.pct_frames_under_pressure <= 0.15:
        _add("possession_pressure", "lightly_pressed", r.pressing_team, int(r.start_frame), int(r.end_frame), int(r.start_frame), bx, by,
             (1 - r.pct_frames_under_pressure) * np.sqrt(r.duration_sec), pct_pressed=float(r.pct_frames_under_pressure), possession_team=int(r.possession_team))

EVENTS = pd.DataFrame(_rows)
EVENTS["clock"] = EVENTS["anchor_frame"].map(clock_str)
EVENTS["uid"] = [f"{f}|{s}|t{t}|{a}" for f, s, t, a in zip(EVENTS.family, EVENTS.subtype, EVENTS.team, EVENTS.anchor_frame)]
EVENTS["team_name"] = EVENTS["team"].map(TEAM_NAMES)
EVENTS = EVENTS[["uid", "family", "subtype", "team", "team_name", "start_frame", "end_frame", "anchor_frame", "clock", "zone", "channel", "target", "score", "payload"]]

print(f"{len(EVENTS)} events in the registry\n")
print(EVENTS.groupby(["family", "subtype", "team_name"]).size().unstack(fill_value=0).to_string())

# ---- table map: where each metric lives (the "structure" of this notebook)
METRIC_TABLES = {
    "pressure signal": ["team_pressure_per_frame", "frame_lane_summary", "team_pressure_full", "team_pressure_events"],
    "zone press (thirds)": ["zone_counts", "zone_pct", "weighted_pressure_seconds"],
    "channel press (left / centre / right)": ["channel_counts", "channel_pct", "zone_channel_counts", "zone_channel_pct"],
    "defensive line": ["line_last1_clean", "line_last3_clean", "line_state", "line_by_ball_zone", "line_by_ball_channel", "defensive_line_events"],
    "team shape": ["team_shape", "team_shape_pressure"],
    "timing / counter-press": ["time_to_press", "turnovers_annotated"],
    "outcomes / effect": ["passes_from_scratch", "defensive_actions", "outcome_events", "zone_summary", "possession_events", "regain_summary"],
    "triggers": ["reception_triggers"],
    "registry + clips": ["EVENTS"],
}
_inv = [dict(dimension=d, table=t, rows=len(globals()[t]) if t in globals() and hasattr(globals()[t], "__len__") else None) for d, ts in METRIC_TABLES.items() for t in ts]
print("\ntable inventory:"); print(pd.DataFrame(_inv).to_string(index=False))


1319 events in the registry

team_name                                 Atletico  Barca
family              subtype                              
counter_press       fast_no_regain              20     26
                    fast_regain                  3      6
                    never                       23     15
                    slow                         8      7
defensive_action    interception                47     43
                    tackle                      19     22
defensive_line      deep_line                   40     20
                    high_line                   33     29
                    line_step_up                10      6
long_ball           lost_after_long_ball         3      7
pass_under_pressure forced_backward             20     18
                    forced_sideways             10     11
                    lost_intercepted            16     17
                    progressed_forward          20      9
possession_pressure heavily_pressed        

## Hand-off
Saves what the later notebooks need.

In [9]:
save_state(globals(), [
    'EVENTS',
    'team_pressing_profile',
], notebook='05_summary_and_registry.ipynb')

saved 2 variables to C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache\pressure_analysis\state
  EVENTS                            DataFrame       0.22 MB
  team_pressing_profile             DataFrame       0.00 MB
